# RunPod QLoRA 4단계 순차 학습 — Qwen3-VL 요구사항 생성 전용 v4

이 노트북은 **대표 기반 모델을 `Qwen/Qwen3-VL-8B-Instruct`로 변경**하고, 기존 4단계 요구사항 생성 학습을 같은 LoRA 어댑터에 순차 적용하는 실행본입니다.

## 이번 파인튜닝의 범위

- 학습 목적: 요구사항 원자분해·결합·정규화·전역 중복제거·CBD 변환
- 학습 데이터: 현재 4TASK **텍스트 JSONL** 그대로 사용
- 비전 인코더 및 비전 결합부: **고정(freeze)**
- LoRA 적용 범위: Qwen3-VL의 **언어 모델 계층만**
- 최종 운영 형태: `Qwen3-VL-8B-Instruct + 4단계 최종 LoRA 어댑터`

즉, 이미지 인식 자체를 새로 학습하지 않습니다. Qwen3-VL의 기존 이미지·표·OCR 이해 능력을 유지하면서, 언어 모델에 요구사항 생성 규칙을 추가합니다.

## 4단계

1. **1단계**: Task1 원자분해 + Task2 결합·정규화 + Task3 그룹판정(AUX)
2. **2단계**: Task4 단건 CBD 변환(AUX)
3. **3단계**: Task3 문서 전체 전역 중복제거·의미병합
4. **4단계**: Task4 문서 전체 최종 요구사항 → CBD

## 장문 설정

- 3단계 상한: **114,688 tokens**
- 4단계 상한: **98,304 tokens**
- 길이 기준: `system + user + assistant 정답` 전체 학습 시퀀스
- 초과 행은 임의 절단하거나 조용히 제외하지 않고 학습 전에 중단

## GPU

- **A100 PCIe 80GB 사용 가능**
- H100 80GB는 필수가 아니라 속도상 권장 사양
- 3·4단계는 80GB급 단일 GPU 기준

> 중요: VARCO 기반 v3의 LoRA 저장소·체크포인트와 호환되지 않습니다. v4는 별도의 Qwen3-VL 저장소명과 로컬 폴더를 사용합니다.


In [1]:
!nvidia-smi

Sun Jun 14 11:52:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200 NVL                On  |   00000000:61:00.0 Off |                  Off |
| N/A   32C    P0             71W /  600W |       0MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. 패키지 설치

Qwen3-VL 공식 Transformers 구현이 포함된 4.57 이상 계열을 사용합니다. 메이저 버전 변경으로 인한 API 불일치를 피하기 위해 Transformers 5 미만으로 제한합니다.

설치가 끝난 뒤 import 오류가 발생하면 Jupyter의 **Kernel → Restart Kernel**을 한 번 실행하고 다시 진행합니다.


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
%pip install -U "transformers>=4.57.0,<5" "peft>=0.17.1,<1" "accelerate>=1.7,<2" datasets bitsandbytes huggingface_hub hf-xet packaging ninja pillow requests
!MAX_JOBS=4 pip install -U flash-attn --no-build-isolation


  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
Using cached peft-0.19.1-py3-none-any.whl (680 kB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [peft]3/4 [peft]ndbytes]
Note: you may need to restart the kernel to use updated packages.
  Using cached flash_attn-2.8.3.post1.tar.gz (8.5 MB)
  Preparing metadata (pyproject.toml) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3.post1-cp312-cp312-linux_x86_64.whl size=256040449 sha256=9a08775a6be3358e3b691ed97f7cb90ad4e9eb6a912e8bce680c2e

In [4]:
%pip install -U liger-kernel

Note: you may need to restart the kernel to use updated packages.


## 2. 사용자 설정

아래의 `내아이디`만 Hugging Face 사용자명 또는 조직명으로 변경합니다.

v3의 VARCO 어댑터와 섞이지 않도록 저장소명이 `req-qwen3vl-*`로 분리되어 있습니다. 기존 VARCO 저장소명을 재사용하지 마세요.


In [5]:
%pip install -q python-dotenv
%pip install hf_transfer

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv("/workspace/env", override=True)

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("/workspace/env에서 HF_TOKEN을 찾지 못했습니다.")

login(
    token=HF_TOKEN,
    add_to_git_credential=False,
)

print("Hugging Face 로그인 완료")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face 로그인 완료


In [7]:
import os
import gc
import time
import json
import inspect
import torch


# ===== 반드시 수정할 값 =====
HF_DATASET_REPO = "jaehoony/requirements-4task-dataset"
BASE_MODEL      = "Qwen/Qwen3-VL-8B-Instruct"

# VARCO용 기존 어댑터 저장소와 반드시 분리
STAGE1_REPO = "jaehoony/req-qwen3vl-stage1-core"
STAGE2_REPO = "jaehoony/req-qwen3vl-stage2-task4aux"
STAGE3_REPO = "jaehoony/req-qwen3vl-stage3-task3doc"
STAGE4_REPO = "jaehoony/req-qwen3vl-stage4-task4final"

# ===== 실행 단계 =====
# stage3 재실행시 False
RUN_STAGE1 = False
RUN_STAGE2 = False
# RUN_STAGE1 = True
# RUN_STAGE2 = True
RUN_STAGE3 = True
RUN_STAGE4 = True

# 3단계를 이미 Hub/로컬에 저장했고 이번 실행에서만 건너뛸 때 True
USE_EXISTING_STAGE3_FOR_STAGE4 = True

# 단계 완료 후 Hugging Face Hub에 어댑터 업로드
PUSH_TO_HUB = True

# 중단된 동일 단계의 /workspace/ckpt/qwen3vl_.../checkpoint-* 자동 재개
AUTO_RESUME = True

# ===== 장문 설정 =====
LONG_CEILING_S3 = 114_688
LONG_CEILING_S4 = 98_304
REQUIRE_ALL_LONG_ROWS = True

# 100K 전후 장문 학습용
ATTN_IMPLEMENTATION = "flash_attention_2"
MIN_LONG_GPU_GB = 75

# ===== QLoRA 설정 =====
LORA_R = 16 # 내일 8로 낮추기 
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LANGUAGE_LORA_LEAF_MODULES = (
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
)

GRAD_ACCUM_SHORT = 8
GRAD_ACCUM_LONG  = 8

# 선택적 최종 테스트. 기본 False라 학습 완료 직후 자동 추론하지 않음
RUN_TEXT_SMOKE_TEST = False
RUN_IMAGE_SMOKE_TEST = False
IMAGE_TEST_PATH = ""  # 예: /workspace/test/page_01.png

# 성능 설정
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")


PyTorch: 2.8.0+cu128
CUDA: 12.8
GPU: NVIDIA H200 NVL


In [8]:
# GPU·Qwen3-VL·FlashAttention 사전 점검
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 감지되지 않습니다. RunPod GPU Pod에서 실행하세요.")

gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 1024**3
print(f"GPU: {gpu.name}")
print(f"GPU memory: {gpu_gb:.1f} GB")

if (RUN_STAGE3 or RUN_STAGE4) and gpu_gb < MIN_LONG_GPU_GB:
    raise RuntimeError(
        f"3·4단계 장문 학습은 80GB급 GPU가 필요합니다. 현재 GPU는 {gpu_gb:.1f}GB입니다."
    )

try:
    import transformers
    import peft
    from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
    print("transformers:", transformers.__version__)
    print("peft:", peft.__version__)
except Exception as exc:
    raise RuntimeError(
        "Qwen3-VL 클래스 import에 실패했습니다. 설치 셀 실행 후 Kernel을 재시작하세요."
    ) from exc

try:
    import flash_attn
    print("flash-attn:", getattr(flash_attn, "__version__", "version 확인 불가"))
except Exception as exc:
    if RUN_STAGE3 or RUN_STAGE4:
        raise RuntimeError(
            "flash-attn import에 실패했습니다. 설치 셀을 다시 실행하거나 Kernel을 재시작하세요."
        ) from exc
    print("[주의] 짧은 단계만 실행하므로 flash-attn 오류를 무시합니다:", exc)

if "A100" in gpu.name:
    print("[확인] A100 80GB 계열로 실행합니다. H100은 필수가 아닙니다.")
elif "H100" in gpu.name:
    print("[확인] H100 80GB 계열로 실행합니다.")
else:
    print("[주의] 80GB급 GPU이나 A100/H100이 아닙니다. 첫 장문 step을 반드시 확인하세요.")


GPU: NVIDIA H200 NVL
GPU memory: 139.8 GB
transformers: 4.57.6
peft: 0.19.1
flash-attn: 2.8.3.post1
[주의] 80GB급 GPU이나 A100/H100이 아닙니다. 첫 장문 step을 반드시 확인하세요.


In [9]:
from huggingface_hub import login
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 3. 4개 configuration 데이터셋 로드

In [10]:
from datasets import get_dataset_config_names
from huggingface_hub import HfApi

print("사용 중인 저장소:", HF_DATASET_REPO)

configs = get_dataset_config_names(
    HF_DATASET_REPO,
    token=HF_TOKEN,
)
print("실제 인식 config:", configs)

api = HfApi(token=HF_TOKEN)
repo_files = api.list_repo_files(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
)

print("\n저장소 파일 목록:")
for file_path in repo_files:
    print(file_path)

사용 중인 저장소: jaehoony/requirements-4task-dataset


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


실제 인식 config: ['default']

저장소 파일 목록:
.gitattributes
README.md
stage1_core/test-00000-of-00001.parquet
stage1_core/train-00000-of-00001.parquet
stage1_core/validation-00000-of-00001.parquet
stage2_task4_aux/test-00000-of-00001.parquet
stage2_task4_aux/train-00000-of-00001.parquet
stage2_task4_aux/validation-00000-of-00001.parquet
stage3_task3_doc/test-00000-of-00001.parquet
stage3_task3_doc/train-00000-of-00001.parquet
stage3_task3_doc/validation-00000-of-00001.parquet
stage4_task4_primary/test-00000-of-00001.parquet
stage4_task4_primary/train-00000-of-00001.parquet
stage4_task4_primary/validation-00000-of-00001.parquet


In [11]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download


def load_stage_parquet(stage_name: str):
    """
    Hugging Face Dataset 저장소의 단계별 Parquet 파일을
    정확한 경로로 다운로드한 후 train/validation/test로 로드한다.
    """
    data_files = {}
    for split_name in ("train", "validation", "test"):
        filename = (
            f"{stage_name}/"
            f"{split_name}-00000-of-00001.parquet"
        )
        local_path = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            filename=filename,
            token=HF_TOKEN,
        )
        data_files[split_name] = local_path
        print(f"[다운로드 완료] {filename}")

    return load_dataset(
        "parquet",
        data_files=data_files,
    )


# 기존 노트북과 동일한 변수명 유지
stage1 = load_stage_parquet("stage1_core")
stage2 = load_stage_parquet("stage2_task4_aux")
stage3 = load_stage_parquet("stage3_task3_doc")
stage4 = load_stage_parquet("stage4_task4_primary")

# 행 수 검증
expected_rows = {
    "1단계": {
        "train": 2190,
        "validation": 459,
        "test": 477,
    },
    "2단계": {
        "train": 811,
        "validation": 170,
        "test": 184,
    },
    "3단계": {
        "train": 15,
        "validation": 3,
        "test": 3,
    },
    "4단계": {
        "train": 15,
        "validation": 3,
        "test": 3,
    },
}

loaded_stages = {
    "1단계": stage1,
    "2단계": stage2,
    "3단계": stage3,
    "4단계": stage4,
}

for stage_label, dataset in loaded_stages.items():
    actual = {
        split: dataset[split].num_rows
        for split in dataset
    }

    print(f"{stage_label}: {actual}")

    if actual != expected_rows[stage_label]:
        raise RuntimeError(
            f"{stage_label} 행 수가 예상값과 다릅니다.\n"
            f"예상: {expected_rows[stage_label]}\n"
            f"실제: {actual}"
        )

print("\n✅ 4단계 데이터셋 로딩 및 행 수 검증 완료")

stage1_core/train-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

[다운로드 완료] stage1_core/train-00000-of-00001.parquet


stage1_core/validation-00000-of-00001.pa(…):   0%|          | 0.00/3.45M [00:00<?, ?B/s]

[다운로드 완료] stage1_core/validation-00000-of-00001.parquet


stage1_core/test-00000-of-00001.parquet:   0%|          | 0.00/3.70M [00:00<?, ?B/s]

...

[다운로드 완료] stage4_task4_primary/test-00000-of-00001.parquet


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

1단계: {'train': 2190, 'validation': 459, 'test': 477}
2단계: {'train': 811, 'validation': 170, 'test': 184}
3단계: {'train': 15, 'validation': 3, 'test': 3}
4단계: {'train': 15, 'validation': 3, 'test': 3}

✅ 4단계 데이터셋 로딩 및 행 수 검증 완료


## 4. 데이터 형식 검증 및 실제 토큰 길이 측정

이번 학습 데이터는 요구사항 생성용 **텍스트 데이터**여야 합니다. 데이터에 이미지·영상 항목이 포함되면 즉시 중단합니다.

문자열 형태의 `content`는 Qwen3-VL의 멀티모달 메시지 형식인 `[{"type":"text", "text":"..."}]`로 내부 변환한 뒤 chat template을 적용합니다.

측정값:

- `prompt_tokens`: system + user
- `completion_tokens`: assistant 정답
- `total_tokens`: 실제 학습 전체 길이


In [12]:
from functools import lru_cache
from transformers import AutoProcessor, AutoConfig
import statistics
import math

processor = AutoProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer = processor.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_config = AutoConfig.from_pretrained(BASE_MODEL, token=HF_TOKEN)
text_config = getattr(base_config, "text_config", base_config)
MODEL_CONTEXT_LIMIT = int(
    getattr(text_config, "max_position_embeddings", 262_144)
)
print("Qwen3-VL text context limit:", f"{MODEL_CONTEXT_LIMIT:,}")


def normalize_messages(messages, require_assistant=True):
    normalized = []
    for message in messages:
        role = message.get("role")
        content = message.get("content")

        if role not in {"system", "user", "assistant"}:
            raise ValueError(f"지원하지 않는 role: {role}")

        if isinstance(content, str):
            content = [{"type": "text", "text": content}]
        elif isinstance(content, list):
            clean_content = []
            for item in content:
                if not isinstance(item, dict):
                    raise ValueError("content 배열 항목은 dict여야 합니다.")
                item_type = item.get("type")
                if item_type in {"image", "image_url", "video", "video_url"}:
                    raise ValueError(
                        "v4 학습 데이터는 요구사항 생성용 텍스트 전용입니다. "
                        f"미디어 항목이 발견되었습니다: {item_type}"
                    )
                if item_type != "text":
                    raise ValueError(f"지원하지 않는 content type: {item_type}")
                text = item.get("text")
                if not isinstance(text, str):
                    raise ValueError("text content의 text 값은 문자열이어야 합니다.")
                clean_content.append({"type": "text", "text": text})
            content = clean_content
        else:
            raise ValueError("message.content는 문자열 또는 text 배열이어야 합니다.")

        normalized.append({"role": role, "content": content})

    if require_assistant and (not normalized or normalized[-1]["role"] != "assistant"):
        raise ValueError("각 학습 행의 마지막 메시지는 assistant 정답이어야 합니다.")

    return normalized


def render_ids(messages, add_generation_prompt):
    normalized = normalize_messages(
        messages, require_assistant=not add_generation_prompt
    )
    ids = tokenizer.apply_chat_template(
        normalized,
        tokenize=True,
        add_generation_prompt=add_generation_prompt,
    )
    if isinstance(ids, torch.Tensor):
        ids = ids.tolist()
    if ids and isinstance(ids[0], list):
        ids = ids[0]
    return ids


@lru_cache(maxsize=20_000)
def _token_parts_cached(messages_json):
    messages = json.loads(messages_json)
    full = render_ids(messages, add_generation_prompt=False)
    prompt = render_ids(messages[:-1], add_generation_prompt=True)

    prefix = 0
    for a, b in zip(full, prompt):
        if a != b:
            break
        prefix += 1

    return len(full), prefix, len(full) - prefix


def token_parts(example):
    key = json.dumps(example["messages"], ensure_ascii=False, sort_keys=True)
    total, prompt, completion = _token_parts_cached(key)
    return {"total": total, "prompt": prompt, "completion": completion}


def token_len(example):
    return token_parts(example)["total"]


def percentile(sorted_values, ratio):
    return sorted_values[round((len(sorted_values) - 1) * ratio)]


def measure_stage(name, dataset):
    split_info = {}
    all_lengths = []

    for split in dataset:
        lengths = sorted(token_len(row) for row in dataset[split])
        if not lengths:
            continue
        all_lengths.extend(lengths)
        split_info[split] = {
            "rows": len(lengths),
            "p50": percentile(lengths, 0.50),
            "p90": percentile(lengths, 0.90),
            "p95": percentile(lengths, 0.95),
            "max": lengths[-1],
            "mean": round(statistics.mean(lengths), 1),
        }

    result = {"name": name, "splits": split_info, "max": max(all_lengths)}
    print("\n", name)
    for split, info in split_info.items():
        print(" ", split, info)
    print(" global max:", f"{result['max']:,}")
    return result


# 전체 데이터의 텍스트 전용 스키마를 먼저 검증
for stage_name, dataset in [
    ("1단계", stage1), ("2단계", stage2),
    ("3단계", stage3), ("4단계", stage4),
]:
    for split in dataset:
        for row_index, row in enumerate(dataset[split]):
            try:
                normalize_messages(row["messages"])
            except Exception as exc:
                raise ValueError(
                    f"{stage_name}/{split}/{row_index} 메시지 형식 오류: {exc}"
                ) from exc
    print(f"[통과] {stage_name}: 텍스트 메시지 스키마 정상")

m1 = measure_stage("1단계", stage1)
m2 = measure_stage("2단계", stage2)
m3 = measure_stage("3단계", stage3)
m4 = measure_stage("4단계", stage4)


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Qwen3-VL text context limit: 262,144
[통과] 1단계: 텍스트 메시지 스키마 정상
[통과] 2단계: 텍스트 메시지 스키마 정상
[통과] 3단계: 텍스트 메시지 스키마 정상
[통과] 4단계: 텍스트 메시지 스키마 정상

 1단계
  train {'rows': 2190, 'p50': 2295, 'p90': 2730, 'p95': 3052, 'max': 8693, 'mean': 2308.5}
  validation {'rows': 459, 'p50': 2307, 'p90': 2597, 'p95': 2791, 'max': 4836, 'mean': 2270.2}
  test {'rows': 477, 'p50': 2293, 'p90': 2694, 'p95': 3231, 'max': 5593, 'mean': 2336.7}
 global max: 8,693

 2단계
  train {'rows': 811, 'p50': 2008, 'p90': 2139, 'p95': 2190, 'max': 2420, 'mean': 2025.9}
  validation {'rows': 170, 'p50': 1985, 'p90': 2051, 'p95': 2103, 'max': 2255, 'mean': 2000.2}
  test {'rows': 184, 'p50': 2028, 'p90': 2118, 'p95': 2142, 'max': 2309, 'mean': 2034.5}
 global max: 2,420

 3단계
  train {'rows': 15, 'p50': 22583, 'p90': 45444, 'p95': 45444, 'max': 103209, 'mean': 31798.2}
  validation {'rows': 3, 'p50': 26765, 'p90': 47904, 'p95': 47904, 'max': 47904, 'mean': 31335}
  test {'rows': 3, 'p50': 19867, 'p90': 73247, 'p95': 73247, 'max':

In [13]:
def print_longest_breakdown(name, dataset):
    longest = None
    for split in dataset:
        for index, row in enumerate(dataset[split]):
            parts = token_parts(row)
            item = {
                "split": split,
                "index": index,
                **parts,
            }
            if longest is None or item["total"] > longest["total"]:
                longest = item
    print(
        f"{name} longest -> split={longest['split']}, row={longest['index']}, "
        f"prompt={longest['prompt']:,}, completion={longest['completion']:,}, "
        f"total={longest['total']:,}"
    )

print_longest_breakdown("3단계", stage3)
print_longest_breakdown("4단계", stage4)


3단계 longest -> split=train, row=9, prompt=31,065, completion=72,144, total=103,209
4단계 longest -> split=train, row=6, prompt=47,907, completion=38,732, total=86,639


## 5. 단계별 MAX_LENGTH 결정 및 사전 검증

- 짧은 단계는 측정 최대값에 256 token 여유를 더합니다.
- 장문 단계는 실제 최대값을 512 단위로 올림하되, 지정한 상한과 모델 context limit을 넘지 않습니다.
- 장문 행이 상한을 초과하면 데이터가 조용히 빠지는 대신 여기서 즉시 오류가 발생합니다.


In [14]:
def round_up(value, bucket=512):
    return int(math.ceil(value / bucket) * bucket)

# context limit 끝까지 꽉 채우지 않도록 최소 512 token 안전 여유
MODEL_SAFE_LIMIT = max(512, MODEL_CONTEXT_LIMIT - 512)

MAX_LEN_S1 = min(round_up(m1["max"] + 256), MODEL_SAFE_LIMIT)
MAX_LEN_S2 = min(round_up(m2["max"] + 256), MODEL_SAFE_LIMIT)
MAX_LEN_S3 = min(round_up(m3["max"]), LONG_CEILING_S3, MODEL_SAFE_LIMIT)
MAX_LEN_S4 = min(round_up(m4["max"]), LONG_CEILING_S4, MODEL_SAFE_LIMIT)

MAX_LENGTHS = {
    "1단계": MAX_LEN_S1,
    "2단계": MAX_LEN_S2,
    "3단계": MAX_LEN_S3,
    "4단계": MAX_LEN_S4,
}
print("MAX_LENGTH:", {k: f"{v:,}" for k, v in MAX_LENGTHS.items()})

def count_over_limit(dataset, max_length):
    result = {}
    for split in dataset:
        over = []
        for index, row in enumerate(dataset[split]):
            length = token_len(row)
            if length > max_length:
                over.append((index, length))
        result[split] = over
    return result

for name, dataset, max_length, strict in [
    ("1단계", stage1, MAX_LEN_S1, True),
    ("2단계", stage2, MAX_LEN_S2, True),
    ("3단계", stage3, MAX_LEN_S3, REQUIRE_ALL_LONG_ROWS),
    ("4단계", stage4, MAX_LEN_S4, REQUIRE_ALL_LONG_ROWS),
]:
    overflow = count_over_limit(dataset, max_length)
    total_over = sum(len(rows) for rows in overflow.values())
    if total_over:
        details = {
            split: [(idx, f"{length:,}") for idx, length in rows]
            for split, rows in overflow.items() if rows
        }
        message = (
            f"{name}: MAX_LENGTH {max_length:,} 초과 행 {total_over}개가 있습니다. "
            f"상세={details}"
        )
        if strict:
            raise ValueError(message)
        print("[주의]", message)
    else:
        print(f"[통과] {name}: 모든 행이 MAX_LENGTH {max_length:,} 이내")


MAX_LENGTH: {'1단계': '9,216', '2단계': '3,072', '3단계': '103,424', '4단계': '87,040'}
[통과] 1단계: 모든 행이 MAX_LENGTH 9,216 이내
[통과] 2단계: 모든 행이 MAX_LENGTH 3,072 이내
[통과] 3단계: 모든 행이 MAX_LENGTH 103,424 이내
[통과] 4단계: 모든 행이 MAX_LENGTH 87,040 이내


In [15]:
# 길이 계산
lengths = []

for i, row in enumerate(stage3["train"]):
    text = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    length = len(tokenizer(text)["input_ids"])
    lengths.append((i, length))

# 길이 내림차순 정렬
lengths.sort(key=lambda x: x[1], reverse=True)

# 확인
for rank, (idx, length) in enumerate(lengths):
    print(f"{rank:2d} | row={idx:2d} | length={length:,}")


idx = lengths[0][0]

row = stage3["train"][idx]

print(idx)

for i, msg in enumerate(row["messages"]):
    print("="*80)
    print("message", i)
    print("role =", msg["role"])
    print(msg["content"][:1000])  # 앞부분만 보기

 0 | row= 9 | length=103,209
 1 | row=13 | length=45,444
 2 | row= 3 | length=45,139
 3 | row= 0 | length=35,849
 4 | row= 8 | length=35,833
 5 | row= 2 | length=32,747
 6 | row= 4 | length=23,580
 7 | row=10 | length=22,583
 8 | row=14 | length=20,322
 9 | row=11 | length=20,108
10 | row= 1 | length=20,087
11 | row= 5 | length=19,997
12 | row=12 | length=18,060
13 | row= 7 | length=17,697
14 | row= 6 | length=16,318
9
message 0
role = system
[공통 목적]
공공·민간 정보화사업 RFP에서 기능 요구사항을 추출하여 다음 4개 학습 Task로 변환한다.

Task 1: FUR 전체 → 의미 기반 원자 요구사항 분해
Task 2: 원자 요구사항 후보 → 1차 결합 및 문장 정규화
Task 3: 문서 전체 Task 2 후보 → 의미중복·부분중복·포함관계 병합 및 최종 문체 교정
Task 4: Task 3 최종 요구사항 → CBD 9개 필드 요구사항 정의서 작성

[절대 원칙]
1. 원문에 없는 기능, 사용자, 시스템, 데이터, 조건, 수치, 법령을 추가하지 않는다.
2. 표의 '정의', '세부내용', '산출정보', '관련 요구사항', '출처', '비고' 등의 라벨 자체를 요구사항으로 생성하지 않는다.
3. 기능 의미를 키워드 개수로 판단하지 않는다. 수행대상, 수행주체, 수행행위, 기능 결과, 검수 증거를 함께 판단한다.
4. 동일한 원문 문구가 참고용으로 여러 위치에 반복되어도 동일 atomic_id를 여러 Task 2 결과의 실질 구성원으로 중복 사용하지 않는다.
5. 참고용 문구는 reference_context_i

## 6. assistant 답변 구간만 loss 계산하는 collator

원문과 system/user prompt는 입력 문맥으로만 사용하고, assistant 정답 구간만 loss를 계산합니다.

현재 학습 데이터는 텍스트 전용이므로 `pixel_values`를 만들지 않습니다. 운영 시 이미지 입력은 최종 Qwen3-VL 기반 모델과 `AutoProcessor`가 처리합니다.


In [16]:
from dataclasses import dataclass
from typing import Any
from torch.nn.utils.rnn import pad_sequence

@dataclass
class AssistantOnlyTextCollator:
    tokenizer: Any
    max_length: int

    def __call__(self, batch):
        input_tensors = []
        label_tensors = []

        for example in batch:
            messages = example["messages"]
            full = render_ids(messages, add_generation_prompt=False)
            prompt = render_ids(messages[:-1], add_generation_prompt=True)

            if len(full) > self.max_length:
                raise ValueError(
                    f"길이 {len(full):,} > MAX_LENGTH {self.max_length:,}"
                )

            prefix = 0
            for a, b in zip(full, prompt):
                if a != b:
                    break
                prefix += 1

            if prefix >= len(full):
                raise ValueError("assistant 정답 토큰 구간을 찾지 못했습니다.")

            input_ids = torch.tensor(full, dtype=torch.long)
            labels = input_ids.clone()
            labels[:prefix] = -100

            input_tensors.append(input_ids)
            label_tensors.append(labels)

        input_ids = pad_sequence(
            input_tensors,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )
        labels = pad_sequence(
            label_tensors,
            batch_first=True,
            padding_value=-100,
        )
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


## 7. Qwen3-VL 언어 계층 전용 QLoRA 설정

`target_modules="all-linear"`는 사용하지 않습니다. 그렇게 하면 비전 계층의 선형 모듈까지 LoRA 대상이 될 수 있기 때문입니다.

v4는 모델을 로드한 뒤 `model.language_model` 아래의 다음 계층만 동적으로 찾아 LoRA를 삽입합니다.

- `q_proj`, `k_proj`, `v_proj`, `o_proj`
- `gate_proj`, `up_proj`, `down_proj`

비전 계층(`model.visual`)은 명시적으로 고정하고, 학습 가능 파라미터 검사에서 비전 계층이 발견되면 즉시 중단합니다.


In [17]:
from transformers import BitsAndBytesConfig
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)


def discover_language_lora_targets(model):
    targets = []
    for name, _module in model.named_modules():
        leaf = name.rsplit(".", 1)[-1]
        if "language_model" in name and leaf in LANGUAGE_LORA_LEAF_MODULES:
            targets.append(name)

    targets = sorted(set(targets))
    if not targets:
        raise RuntimeError(
            "Qwen3-VL 언어 모델 LoRA 대상 모듈을 찾지 못했습니다. "
            "Transformers 버전과 모델 구조를 확인하세요."
        )

    invalid = [
        name for name in targets
        if "visual" in name or "vision" in name or "merger" in name
    ]
    if invalid:
        raise RuntimeError(f"비전 모듈이 LoRA 대상에 포함되었습니다: {invalid[:10]}")

    print(f"언어 모델 LoRA 대상: {len(targets)}개")
    print("대상 예시:", targets[:10])
    return targets


def build_language_lora_config(model):
    targets = discover_language_lora_targets(model)
    return LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=targets,
        task_type="CAUSAL_LM",
    )


def freeze_visual_components(model):
    frozen = 0
    for name, parameter in model.named_parameters():
        if "visual" in name or "vision" in name:
            parameter.requires_grad = False
            frozen += parameter.numel()
    print(f"비전 계층 고정 파라미터 수: {frozen:,}")


def assert_language_only_trainable(model):
    trainable = [name for name, p in model.named_parameters() if p.requires_grad]
    if not trainable:
        raise RuntimeError("학습 가능한 LoRA 파라미터가 없습니다.")

    forbidden = [
        name for name in trainable
        if "visual" in name or "vision" in name or "merger" in name
    ]
    if forbidden:
        raise RuntimeError(
            "비전 계층에 학습 가능한 파라미터가 발견되었습니다: "
            f"{forbidden[:20]}"
        )

    non_lora = [name for name in trainable if "lora_" not in name]
    if non_lora:
        raise RuntimeError(
            "LoRA 외 학습 가능 파라미터가 발견되었습니다: "
            f"{non_lora[:20]}"
        )

    print(f"학습 가능 파라미터 텐서: {len(trainable)}개")
    print("학습 범위 예시:", trainable[:10])


In [18]:
# MAX_LENGTH 초과 행 처리
# 긴급 실행본은 기본적으로 초과 행을 조용히 제외하지 않습니다.
def prepare_dataset_by_length(dataset, max_length, name, strict=True):
    from datasets import DatasetDict

    output = {}
    for split in dataset:
        before = dataset[split].num_rows
        kept = dataset[split].filter(
            lambda example: token_len(example) <= max_length,
            desc=f"{name}/{split} 길이 검증",
        )
        dropped = before - kept.num_rows

        if dropped and strict:
            raise ValueError(
                f"{name}/{split}: MAX_LENGTH {max_length:,} 초과로 "
                f"{dropped}행이 제외될 예정이므로 학습을 중단합니다."
            )
        if dropped:
            print(
                f"[주의] {name}/{split}: 길이 초과 {dropped}행 제외 "
                f"({before}->{kept.num_rows})"
            )
        else:
            print(f"[통과] {name}/{split}: {before}행 모두 유지")

        output[split] = kept

    return DatasetDict(output)


## 8. 순차 학습 함수

각 단계마다 원본 `Qwen3-VL-8B-Instruct`를 4bit로 로드한 뒤:

- 1단계: 언어 모델에 새 LoRA 생성
- 2~4단계: 이전 단계 LoRA를 `is_trainable=True`로 로드
- 비전 계층은 모든 단계에서 고정
- 완료 시 LoRA 어댑터와 Qwen3-VL processor를 저장

최종 결과는 전체 기반 모델 복사본이 아니라 **요구사항 생성용 LoRA 어댑터**입니다.


In [19]:
from transformers import (
    Qwen3VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint


def resolve_adapter(hub_repo, local_dir):
    return hub_repo if PUSH_TO_HUB else local_dir


def get_model_context_limit(model):
    cfg = model.config
    text_cfg = getattr(cfg, "text_config", cfg)
    return int(getattr(text_cfg, "max_position_embeddings", MODEL_CONTEXT_LIMIT))


def build_training_args(
    *, output_dir, hub_id, epochs, gradient_accumulation_steps,
    learning_rate, stage_name,
):
    kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=epochs,
    
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=gradient_accumulation_steps,
    
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    
        optim="paged_adamw_8bit",
    
        # 진행률 및 학습 로그
        logging_strategy="steps",
        logging_steps=1 if stage_name in ("3단계", "4단계") else 10,
        logging_first_step=True,
        disable_tqdm=False,
    
        eval_accumulation_steps=1,
    
        save_strategy="epoch",
        save_total_limit=2,
        save_safetensors=True,
        load_best_model_at_end=True,
    
        learning_rate=learning_rate,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
    
        bf16=True,
        tf32=True,
        max_grad_norm=0.3,
    
        remove_unused_columns=False,
    
        push_to_hub=PUSH_TO_HUB,
        hub_model_id=hub_id if PUSH_TO_HUB else None,
    
        report_to="none",
        use_liger_kernel=True,
    )

    # Transformers 버전별 인자명 차이 대응
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"

    return TrainingArguments(**kwargs)


def load_qwen3vl_base_for_training():
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        device_map={"": 0},
        dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPLEMENTATION,
        low_cpu_mem_usage=True,
        token=HF_TOKEN,
    )

    model.config.use_cache = False
    if hasattr(model.config, "text_config"):
        model.config.text_config.use_cache = False

    # k-bit 학습 준비: 기반 가중치 고정 + 입력 gradient 준비
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )
    freeze_visual_components(model)
    return model


def train_stage(
    *,
    stage_name,
    dataset,
    output_dir,
    hub_id,
    max_length,
    learning_rate,
    epochs,
    gradient_accumulation_steps,
    previous_adapter=None,
):
    started_at = time.perf_counter()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    base_model = load_qwen3vl_base_for_training()

    if previous_adapter:
        print(f"[{stage_name}] 이전 Qwen3-VL 어댑터 로드:", previous_adapter)
        model = PeftModel.from_pretrained(
            base_model,
            previous_adapter,
            is_trainable=True,
            token=HF_TOKEN,
        )
    else:
        print(f"[{stage_name}] Qwen3-VL 기반에서 새 언어 LoRA 시작:", BASE_MODEL)
        lora_config = build_language_lora_config(base_model)
        model = get_peft_model(base_model, lora_config)

    assert_language_only_trainable(model)
    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()

    model_limit = get_model_context_limit(model)
    if max_length > model_limit:
        raise ValueError(
            f"{stage_name}: MAX_LENGTH {max_length:,}가 Qwen3-VL text context limit "
            f"{model_limit:,}를 초과합니다."
        )

    collator = AssistantOnlyTextCollator(
        tokenizer=tokenizer,
        max_length=max_length,
    )

    args = build_training_args(
        output_dir=output_dir,
        hub_id=hub_id,
        epochs=epochs,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        stage_name=stage_name,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=collator,
    )

    resume_checkpoint = None
    if AUTO_RESUME and os.path.isdir(output_dir):
        resume_checkpoint = get_last_checkpoint(output_dir)

    if resume_checkpoint:
        print(f"[{stage_name}] 체크포인트 자동 재개:", resume_checkpoint)
    else:
        print(f"[{stage_name}] 새 단계 학습 시작")

    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)

    metadata = {
        "base_model": BASE_MODEL,
        "adapter_stage": stage_name,
        "training_scope": "requirements_generation_text_only",
        "vision_frozen": True,
        "lora_scope": "language_model_only",
        "max_length": max_length,
    }
    with open(os.path.join(output_dir, "requirements_training_metadata.json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    if PUSH_TO_HUB:
        trainer.push_to_hub()

    elapsed_hours = (time.perf_counter() - started_at) / 3600
    peak_allocated = torch.cuda.max_memory_allocated() / 1024**3
    peak_reserved = torch.cuda.max_memory_reserved() / 1024**3

    summary = {
        "stage": stage_name,
        "base_model": BASE_MODEL,
        "output_dir": output_dir,
        "hub_id": hub_id if PUSH_TO_HUB else None,
        "vision_frozen": True,
        "max_length": max_length,
        "elapsed_hours": round(elapsed_hours, 3),
        "peak_allocated_gb": round(peak_allocated, 2),
        "peak_reserved_gb": round(peak_reserved, 2),
        "train_metrics": train_result.metrics,
    }
    print("\n[단계 완료]", summary)

    del trainer
    del model
    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    return summary


## 9. 1단계 — 핵심 판단(Task1·2·3 AUX)

Qwen3-VL 언어 모델에 최초 요구사항 생성 LoRA를 생성합니다. 비전 계층은 고정됩니다.

- epoch 2가 적은 것이 아니라, 데이터 규모와 토큰 길이를 고려해 유효 학습량을 계산한 결과

In [20]:
S1_LOCAL = "/workspace/ckpt/qwen3vl_1단계_core"

if RUN_STAGE1:
    s1 = prepare_dataset_by_length(stage1, MAX_LEN_S1, "1단계", strict=True)
    result_s1 = train_stage(
        stage_name="1단계",
        dataset=s1,
        output_dir=S1_LOCAL,
        hub_id=STAGE1_REPO,
        max_length=MAX_LEN_S1,
        learning_rate=1e-4,
        epochs=2,
        gradient_accumulation_steps=GRAD_ACCUM_SHORT,
        previous_adapter=None,
    )
else:
    print("[건너뜀] 1단계")


[건너뜀] 1단계


## 10. 2단계 — Task4 단건 CBD(AUX)

In [21]:
S2_LOCAL = "/workspace/ckpt/qwen3vl_2단계_task4aux"
PREV_FOR_S2 = resolve_adapter(STAGE1_REPO, S1_LOCAL)

if RUN_STAGE2:
    s2 = prepare_dataset_by_length(stage2, MAX_LEN_S2, "2단계", strict=True)
    result_s2 = train_stage(
        stage_name="2단계",
        dataset=s2,
        output_dir=S2_LOCAL,
        hub_id=STAGE2_REPO,
        max_length=MAX_LEN_S2,
        learning_rate=5e-5,
        epochs=2,
        gradient_accumulation_steps=GRAD_ACCUM_SHORT,
        previous_adapter=PREV_FOR_S2,
    )
else:
    print("[건너뜀] 2단계")


[건너뜀] 2단계


## 11. 3단계 — 문서 전체 전역 중복제거·의미병합

- 기본 상한: 114,688
- 전체 학습 시퀀스가 상한 이내인지 사전 검증
- 초과 행은 자동 제외하지 않고 즉시 중단
- A100 PCIe 80GB 또는 H100 80GB 사용
- Qwen3-VL 비전 계층은 계속 고정


In [22]:
S3_LOCAL = "/workspace/ckpt/qwen3vl_3단계_task3doc"
PREV_FOR_S3 = resolve_adapter(STAGE2_REPO, S2_LOCAL)
GRAD_ACCUM_LONG = 1 # 30 update
if RUN_STAGE3:
    s3 = prepare_dataset_by_length(
        stage3,
        MAX_LEN_S3,
        "3단계",
        strict=REQUIRE_ALL_LONG_ROWS,
    )
    result_s3 = train_stage(
        stage_name="3단계",
        dataset=s3,
        output_dir=S3_LOCAL,
        hub_id=STAGE3_REPO,
        max_length=MAX_LEN_S3,
        learning_rate=2e-5,
        epochs=5,
        gradient_accumulation_steps=GRAD_ACCUM_LONG,
        previous_adapter=PREV_FOR_S3,
    )
else:
    print("[건너뜀] 3단계")


3단계/train 길이 검증:   0%|          | 0/15 [00:00<?, ? examples/s]

[통과] 3단계/train: 15행 모두 유지


3단계/validation 길이 검증:   0%|          | 0/3 [00:00<?, ? examples/s]

[통과] 3단계/validation: 3행 모두 유지


3단계/test 길이 검증:   0%|          | 0/3 [00:00<?, ? examples/s]

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


[단계 완료] {'stage': '3단계', 'base_model': 'Qwen/Qwen3-VL-8B-Instruct', 'output_dir': '/workspace/ckpt/qwen3vl_3단계_task3doc', 'hub_id': 'jaehoony/req-qwen3vl-stage3-task3doc', 'vision_frozen': True, 'max_length': 103424, 'elapsed_hours': 1.032, 'peak_allocated_gb': 110.66, 'peak_reserved_gb': 112.83, 'train_metrics': {'train_runtime': 1425.1848, 'train_samples_per_second': 0.053, 'train_steps_per_second': 0.053, 'total_flos': 1.1713197587332752e+17, 'train_loss': 0.024195521076520284, 'epoch': 5.0}}


## 12. 4단계 — 문서 전체 최종 요구사항 → CBD

실제 운영 입출력 형식을 최종 보정합니다.

- 기본 상한: 98,304
- 3단계를 이번 실행에서 완료하거나 기존 저장소를 사용하면 3단계 어댑터에서 이어서 학습
- `USE_EXISTING_STAGE3_FOR_STAGE4=False`이면 2단계 어댑터에서 이어서 학습


In [23]:
S4_LOCAL = "/workspace/ckpt/qwen3vl_4단계_task4final"
GRAD_ACCUM_LONG = 1 # 15 update
if RUN_STAGE3 or USE_EXISTING_STAGE3_FOR_STAGE4:
    PREV_FOR_S4 = resolve_adapter(STAGE3_REPO, S3_LOCAL)
else:
    PREV_FOR_S4 = resolve_adapter(STAGE2_REPO, S2_LOCAL)

if RUN_STAGE4:
    s4 = prepare_dataset_by_length(
        stage4,
        MAX_LEN_S4,
        "4단계",
        strict=REQUIRE_ALL_LONG_ROWS,
    )
    result_s4 = train_stage(
        stage_name="4단계",
        dataset=s4,
        output_dir=S4_LOCAL,
        hub_id=STAGE4_REPO,
        max_length=MAX_LEN_S4,
        learning_rate=1e-5,
        epochs=3,
        gradient_accumulation_steps=GRAD_ACCUM_LONG,
        previous_adapter=PREV_FOR_S4,
    )
else:
    print("[건너뜀] 4단계")


4단계/train 길이 검증:   0%|          | 0/15 [00:00<?, ? examples/s]

[통과] 4단계/train: 15행 모두 유지


4단계/validation 길이 검증:   0%|          | 0/3 [00:00<?, ? examples/s]

[통과] 4단계/validation: 3행 모두 유지


4단계/test 길이 검증:   0%|          | 0/3 [00:00<?, ? examples/s]

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


[단계 완료] {'stage': '4단계', 'base_model': 'Qwen/Qwen3-VL-8B-Instruct', 'output_dir': '/workspace/ckpt/qwen3vl_4단계_task4final', 'hub_id': 'jaehoony/req-qwen3vl-stage4-task4final', 'vision_frozen': True, 'max_length': 87040, 'elapsed_hours': 0.111, 'peak_allocated_gb': 96.59, 'peak_reserved_gb': 98.62, 'train_metrics': {'train_runtime': 380.3787, 'train_samples_per_second': 0.118, 'train_steps_per_second': 0.118, 'total_flos': 5.544000793258733e+16, 'train_loss': 0.020663010432488388, 'epoch': 3.0}}


## 13. 최종 확인 및 선택적 텍스트·이미지 스모크 테스트

최종 운영 조합:

- 기반 모델: `Qwen/Qwen3-VL-8B-Instruct`
- 최종 어댑터: `STAGE4_REPO`
- RunPod 로컬: `/workspace/ckpt/qwen3vl_4단계_task4final`

저장되는 LoRA는 요구사항 생성 능력만 담고, 이미지 인식 능력은 원본 Qwen3-VL 기반 모델이 제공합니다. 실제 서비스에서는 기반 모델과 최종 LoRA를 함께 로드합니다.

### 오류별 즉시 대응

1. **Qwen3VLForConditionalGeneration import 실패**: 설치 셀 재실행 후 Kernel 재시작
2. **비전 모듈이 LoRA 대상이라는 오류**: 학습 중단, `all-linear`로 임의 변경하지 않기
3. **3·4단계 첫 step OOM**: 다른 GPU 프로세스 종료 후 새 Pod에서 재시도
4. **기존 어댑터 로드 오류**: `req-qwen3vl-*` 저장소인지 확인
5. **Pod 종료**: 동일 Network Volume에서 `AUTO_RESUME=True`로 재실행

다음 셀은 기본적으로 실행되지 않습니다. 학습 완료 후 `RUN_TEXT_SMOKE_TEST` 또는 `RUN_IMAGE_SMOKE_TEST`를 `True`로 설정했을 때만 최종 어댑터를 로드합니다.

> 테스트셋은 최종 4단계 체크포인트 확정 후 한 번만 사용합니다.


In [24]:
from peft import PeftModel

FINAL_ADAPTER = resolve_adapter(STAGE4_REPO, S4_LOCAL)

if RUN_TEXT_SMOKE_TEST or RUN_IMAGE_SMOKE_TEST:
    smoke_base = Qwen3VLForConditionalGeneration.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        device_map={"": 0},
        dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPLEMENTATION,
        token=HF_TOKEN,
    )
    smoke_model = PeftModel.from_pretrained(
        smoke_base,
        FINAL_ADAPTER,
        token=HF_TOKEN,
    ).eval()

    def generate_qwen3vl(messages, max_new_tokens=512):
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs.pop("token_type_ids", None)
        inputs = {
            key: value.to(smoke_model.device) if hasattr(value, "to") else value
            for key, value in inputs.items()
        }
        with torch.inference_mode():
            generated = smoke_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
            )
        trimmed = [
            out[len(inp):]
            for inp, out in zip(inputs["input_ids"], generated)
        ]
        return processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

    if RUN_TEXT_SMOKE_TEST:
        text_messages = [{
            "role": "user",
            "content": [{
                "type": "text",
                "text": (
                    "다음 원문을 원자 요구사항으로 분해하라: "
                    "사용자는 공지사항을 조회하고 관리자는 공지사항을 "
                    "등록·수정·삭제할 수 있어야 한다."
                ),
            }],
        }]
        print("[텍스트 테스트]", generate_qwen3vl(text_messages))

    if RUN_IMAGE_SMOKE_TEST:
        if not IMAGE_TEST_PATH:
            raise ValueError("IMAGE_TEST_PATH에 실제 이미지 경로를 지정하세요.")
        image_messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": IMAGE_TEST_PATH},
                {
                    "type": "text",
                    "text": "이 이미지의 표와 텍스트 내용을 요약하고 요구사항 후보를 작성하라.",
                },
            ],
        }]
        print("[이미지 테스트]", generate_qwen3vl(image_messages))
else:
    print("[건너뜀] 최종 스모크 테스트")


[건너뜀] 최종 스모크 테스트



본 학습에서는 단계별 데이터 규모와 시퀀스 길이가 상이하므로 epoch 수를 일률적으로 설정하지 않았다. 

학습 샘플 수, 평균·최대 시퀀스 길이, optimizer update 횟수, 단계별 역할과 과적합 가능성을 함께 고려하여 결정하였다.

1단계는 2,190개 샘플을 2 epoch 학습하여 약 1,011만 토큰이 노출되고 약 548회의 optimizer update가 수행된다. 2단계는 811개 샘플을 2 epoch 학습하여 약 329만 토큰과 약 204회의 update를 확보한다.

3·4단계는 각각 15개의 문서로 구성되지만 평균 약 3.2만 토큰과 2.5만 토큰의 장문 데이터이다. 장문 단계에서는 gradient accumulation을 1로 적용하고, 3단계는 5 epoch로 약 75회, 4단계는 3 epoch로 약 45회의 optimizer update를 수행하도록 설정하였다. 이를 통해 문서 전체 입출력 형식을 충분히 보정하면서도 동일한 소수 문서를 과도하게 반복하여 암기하는 위험을 줄였다.


# ===== QLoRA 설정 수정 제안 =====
LORA_R = 8              # 16에서 낮춤
LORA_ALPHA = 16         # 32에서 낮춤
LORA_DROPOUT = 0.1      # 0.05에서 높임
